# TP 2: Text Processing, Bag-of-Words and N-gram Language Models

In this TP you build, in Python, the text-processing pipeline that turns raw text into index terms, then use it to build **bag-of-words** vectors, an **inverted index** and a **bigram language model**.

| Exercise | Topic | Points |
|---|---|---|
| 1 | Tokenization with regular expressions | 15 |
| 2 | Normalization, stop words, stemming and lemmatization | 25 |
| 3 | Subword tokenization (Hugging Face) | 10 |
| 4 | Bag-of-words and the inverted index | 20 |
| 5 | Bigram language model, add-k smoothing, perplexity | 30 |

* Complete every cell marked `# TODO`, and write short answers in the **Answer** cells.
* Submit **only this notebook**, renamed `TP2_StudentID_Name_Group.ipynb` for example `TP2_e202000129_DORN_Dawin_I5AMS_A.ipynb`, by email (my email: dorn.dawin@gmail.com) **[25/05/2026 11:59 PM]**.

## 0 · Setup
Run these two cells first. On Google Colab everything is already installed; the first cell only downloads NLTK word lists.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

import re, math
from collections import Counter
import pandas as pd

In [ ]:
# Our small collection: 12 short documents about search engines
DOCS = [
    "Search engines index billions of web pages every day.",
    "A search engine ranks the indexed pages for each query.",
    "Users type short queries into the search box.",
    "The crawler downloads web pages and follows their links.",
    "Google's crawler found 3 new pages in the U.S.A. yesterday.",
    "Indexing makes searching fast, but building the index takes time.",
    "The inverted index maps each term to the documents that contain it.",
    "Users don't read every result; they click the top-ranked pages.",
    "Ranking models compare the query with each indexed document.",
    "Stemming maps searching, searched and searches to one term.",
    "Stop words such as 'the' and 'of' are very frequent.",
    "Language models assign a probability to every query.",
]
for i, d in enumerate(DOCS):
    print(i, d)

## Exercise 1 · Tokenization (15 pts)
**Tokenization** splits a text into tokens (words, numbers, punctuation marks). It is the first step of the text transformation component of a search engine (Lecture 1).

### 1.1 · Splitting on spaces (3 pts)
Run the cell. Python's `str.split()` splits on whitespace only.

In [ ]:
print(DOCS[4].split())
print(DOCS[7].split())

**Question 1.1.** Name two problems you see in this output.

**Answer 1.1:** *(write your answer here)*

### 1.2 · A regular-expression tokenizer (6 pts)
The pattern below is adapted from the NLTK example in Jurafsky & Martin, *Speech and Language Processing*, section 2.8. The alternatives are tried **from left to right**.

Complete `tokenize(text)` so that it returns the list of all matches of `PATTERN` in `text` (hint: `re.findall`).

In [ ]:
PATTERN = r"""(?x)
    (?:[A-Z]\.)+            # abbreviations: U.S.A.
  | \w+(?:[-']\w+)*         # words, with optional internal hyphen or apostrophe
  | \$?\d+(?:\.\d+)?%?      # numbers, money, percentages
  | \.\.\.                  # ellipsis
  | [][.,;"'?():_`-]        # single punctuation marks
"""

def tokenize(text):
    # TODO: return the list of tokens
    ...

for i in (4, 7, 10):
    print(tokenize(DOCS[i]))

**Question 1.2.** (a) Compare with 1.1: what is fixed? (b) Why must the abbreviation alternative `(?:[A-Z]\.)+` come **before** the word alternative `\w+…`? (Try swapping them.)

**Answer 1.2:** *(write your answer here)*

### 1.3 · Tokens and types (6 pts)
* A **token** is one occurrence of a word; a **type** is a distinct word (an entry of the vocabulary).
* Tokenize all 12 documents, lower-case every token, and keep only tokens that contain a letter or a digit (drop pure punctuation).
* Print the number of tokens N, the number of types |V|, and the 10 most frequent tokens (`Counter.most_common`).

In [ ]:
# TODO
tokens = ...            # list of lower-cased word tokens of the whole collection
N = ...
V = ...
print("N =", N, " |V| =", V)
print(Counter(tokens).most_common(10))

**Question 1.3.** Which kind of word is the most frequent? Why is that a problem for an index?

**Answer 1.3:** *(write your answer here)*

## Exercise 2 · Normalization, stop words, stemming (25 pts)
**Normalization** maps tokens that should match to the same term, e.g. *Search*, *search* → *search*. **Stemming** and **lemmatization** reduce word variants (*searching*, *searches*) to a common form.

### 2.1 · Normalization (5 pts)
Complete `normalize(tokens)` with these rules, in this order:
1. lower-case every token;
2. drop tokens that contain no letter and no digit (punctuation);
3. remove a final possessive `'s`: `google's` → `google`;
4. remove the dots of abbreviations: `u.s.a.` → `usa`.

**Check:** `['google', 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'usa', 'yesterday']`

In [ ]:
def normalize(tokens):
    out = []
    for t in tokens:
        # TODO: apply rules 1-4, append the result to out
        ...
    return out

print(normalize(tokenize(DOCS[4])))

### 2.2 · Stop words (5 pts)
Use NLTK's English stop list. Remove stop words from the normalized tokens of the **whole collection** and report the number of tokens before and after, and the percentage removed.

In [ ]:
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))
print(len(STOP), "stop words, e.g.", sorted(STOP)[:10])

# TODO
all_terms = ...       # normalized tokens of all documents
kept = ...            # without stop words
print(len(all_terms), "->", len(kept), f"({100 * (1 - len(kept) / len(all_terms)):.1f}% removed)")

**Question 2.2.** Give one search query that would be damaged by removing stop words, and explain why.

**Answer 2.2:** *(write your answer here)*

### 2.3 · Porter vs Snowball vs lemmatization (8 pts)
Fill a table with one row per word and the columns: `porter`, `snowball`, `lemma_noun` (WordNet lemmatizer, default part of speech = noun) and `lemma_verb` (`pos="v"`).

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
porter, snowball, wnl = PorterStemmer(), SnowballStemmer("english"), WordNetLemmatizer()

WORDS = ["searching", "searched", "searches", "indexing", "engines", "queries", "users",
         "libraries", "generously", "university", "universe", "organization", "organ",
         "running", "feet", "was"]

rows = []
for w in WORDS:
    # TODO: append a dict with keys word, porter, snowball, lemma_noun, lemma_verb
    ...
pd.DataFrame(rows)

**Question 2.3.** (a) Give two stems that are not real English words. (b) Give one pair of words with **different meanings** that Porter maps to the same stem, and explain why this hurts precision. (c) Why does `wnl.lemmatize("searching")` return *searching*, but `pos="v"` returns *search*? (d) Give one word where Porter and Snowball disagree.

**Answer 2.3:** *(write your answer here)*

### 2.4 · Your `preprocess` function (7 pts)
Write `preprocess(text, stemmer="porter", remove_stop=True)` that runs the whole pipeline with the functions you wrote above:
`tokenize` → `normalize` → (optional) stop-word removal → stemming (`"porter"`, `"snowball"`, `"lemma"` or `None`), and returns the list of terms.

Later TPs start from this function, so keep it clean: you will copy this cell (possibly updated) into the next notebooks.

**Check:** `preprocess(DOCS[9])` → `['stem', 'map', 'search', 'search', 'search', 'one', 'term']`

In [ ]:
def preprocess(text, stemmer="porter", remove_stop=True):
    # TODO: tokenize -> normalize -> stop words -> stemmer (use porter, snowball, wnl from 2.3)
    ...

for i in (0, 4, 9):
    print(preprocess(DOCS[i]))
print(preprocess(DOCS[9], stemmer="lemma"))
print(preprocess(DOCS[9], stemmer=None, remove_stop=False))

**Question 2.4.** Compare the Porter and lemma outputs for document 9. Which one would let the query *search* match this document best, and why?

**Answer 2.4:** *(write your answer here)*

## Exercise 3 · Subword tokenization (10 pts)
Neural models do not use a fixed list of words. They split text into **subwords** learned from data: frequent words stay whole, rare words are split into frequent pieces, so there are no unknown words. BERT uses the **WordPiece** algorithm; GPT models use **BPE** (byte-pair encoding).

### 3.1 · Load a pretrained tokenizer (4 pts)
Check that you obtain `['i', 'have', 'a', 'new', 'gp', '##u', '!']`.

In [ ]:
from transformers import AutoTokenizer
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
print(bert.tokenize("I have a new GPU!"))

### 3.2 · Explore (6 pts)
Tokenize each word of the list and document 4. For each, print the subword tokens and their number. Then add **three words of your own** (e.g. a rare word, a misspelling, a place name).

In [ ]:
TEST_WORDS = ["search", "searching", "tokenization", "crawler", "bigram", "perplexity", "serach"]
# TODO: add three words of your own, then print each word, its subword tokens and their number
...
# TODO: tokenize DOCS[4] with bert and compare the number of tokens with tokenize(DOCS[4])
...

**Question 3.2.** (a) What does the prefix `##` mean? (b) Which kind of words are split into several pieces, and which stay whole? Why? (c) Why can a subword tokenizer never produce an *unknown word*?

**Answer 3.2:** *(write your answer here)*

## Exercise 4 · Bag-of-words and the inverted index (20 pts)
In the **bag-of-words** model a document is represented by the counts of its terms; word order is ignored.

### 4.1 · Document-term matrix from scratch (8 pts)
1. Preprocess every document with your `preprocess` (Porter, stop words removed).
2. Build the vocabulary: the **sorted** list of all distinct terms.
3. Build the matrix `M` (a list of lists): `M[i][j]` = number of times term `vocab[j]` occurs in document `i`.
4. Print the shape, the percentage of zero cells, and the non-zero entries of document 0.

**Check:** the matrix has 12 rows and 48 columns.

In [ ]:
# TODO
processed = ...
vocab = ...
M = ...

zeros = ...
print("shape:", len(M), "x", len(vocab), f"| zeros: {zeros:.1f}%")
print({vocab[j]: c for j, c in enumerate(M[0]) if c > 0})

### 4.2 · Check with scikit-learn (4 pts)
`CountVectorizer` builds the same matrix. Give it your own pipeline with `analyzer=preprocess`, then check that the vocabulary and the matrix are identical to yours.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# TODO
cv = ...
X = ...
print("same vocabulary:", ...)
print("same matrix:", ...)

### 4.3 · From the matrix to an inverted index (5 pts)
Build a dictionary `index` that maps each term to its postings list `[(doc_id, tf), ...]`, reading **one column** of `M` per term and keeping only the non-zero cells. Print the postings of `search`, `index`, `page` and `crawler`.

In [ ]:
# TODO
index = ...
for t in ["search", "index", "page", "crawler"]:
    print(t, "->", index[t])

**Question 4.3.** (a) Why does scikit-learn return a *sparse* matrix instead of a normal list of lists? (b) What is the relation between a column of `M` and a postings list?

**Answer 4.3:** *(write your answer here)*

### 4.4 · The limit of bag-of-words (3 pts)
Write **two short sentences with different meanings** that get exactly the same bag-of-words vector, and show it with code.

In [ ]:
# TODO
s1, s2 = ..., ...
print(Counter(preprocess(s1)) == Counter(preprocess(s2)))

**Answer 4.4:** *(write your answer here)*

## Exercise 5 · A bigram language model (30 pts)
A bigram model estimates P(w | previous word) from counts and multiplies these probabilities along a sentence. For language models we **keep stop words and do not stem**: function words matter for word order.

### 5.1 · Counting (6 pts)
* `lm_tokens(text)`: the lower-cased word tokens of `text` (use `tokenize`, drop pure punctuation).
* Training sentences: each document becomes `["<s>"] + lm_tokens(doc) + ["</s>"]`.
* Count bigrams in `bigram_c`, and in `context_c` count each word **as the first word of a bigram** (every token except the last `</s>`).
* `V` = number of distinct word types in the training sentences, **including `</s>` but not `<s>`**.

**Check:** V = 78, 122 bigram types, C(&lt;s&gt;) = 12.

In [ ]:
def lm_tokens(text):
    # TODO
    ...

train = [["<s>"] + lm_tokens(d) + ["</s>"] for d in DOCS]
bigram_c, context_c = Counter(), Counter()
for s in train:
    # TODO: update both counters
    ...
V = ...
print("V =", V, "| bigram types:", len(bigram_c), "| C(<s>) =", context_c["<s>"])
print(train[3])

### 5.2 · MLE probabilities (6 pts)
P_MLE(w | prev) = C(prev w) / C(prev). Write `p_mle(w, prev)` and `sentence_prob(text, p)`, which pads the sentence with `<s>` and `</s>` and multiplies the bigram probabilities given by the function `p`. Compute the probability of the three sentences.

In [ ]:
def p_mle(w, prev):
    # TODO
    ...

def sentence_prob(text, p):
    # TODO
    ...

for s in ["the crawler downloads web pages",
          "the crawler downloads web pages every day",
          "pages web downloads crawler the"]:
    print(f"{sentence_prob(s, p_mle):.6f}  {s}")

**Question 5.2.** The second sentence is not in the collection. Why is its probability not zero? Why is the third one zero although it uses the same words as the first?

**Answer 5.2:** *(write your answer here)*

### 5.3 · Add-k smoothing (6 pts)
P_add-k(w | prev) = (C(prev w) + k) / (C(prev) + k·V). Write `p_addk(w, prev, k)` and check that P(engine | search) = 2/81 ≈ 0.0247 for k = 1.

In [ ]:
def p_addk(w, prev, k):
    # TODO
    ...

print(p_mle("engine", "search"), p_addk("engine", "search", 1), p_addk("engine", "search", 0.1))
print(sentence_prob("pages web downloads crawler the", lambda w, prev: p_addk(w, prev, 1)))

**Question 5.3.** P(engine | search) is 1/3 with MLE but only 0.025 with k = 1. Where did the probability go?

**Answer 5.3:** *(write your answer here)*

### 5.4 · Perplexity on held-out sentences (8 pts)
PP = exp(−(1/N) Σ log P(w | prev)), where N counts every predicted token, including `</s>`. Use **log probabilities** (sum of logs) to avoid underflow; return `float("inf")` if a probability is 0.

Compute the perplexity of the held-out sentences with MLE and with add-k for k ∈ {1, 0.5, 0.1, 0.05, 0.01, 0.001}, and show the results in a table.

In [ ]:
HELD_OUT = ["the search engine ranks pages for users",
            "users click the top pages",
            "the crawler follows links to new pages"]

def perplexity(sentences, p):
    # TODO
    ...

print("MLE:", perplexity(HELD_OUT, p_mle))
# TODO: table of k and perplexity

**Question 5.4.** (a) Why is the MLE perplexity infinite? (b) Which k gives the lowest perplexity? (c) Explain why a **large** k and a **very small** k are both worse.

**Answer 5.4:** *(write your answer here)*